# FarmaTEF · comparación de detectores

El cuaderno compara YOLOv8n y YOLO11n para localizar objetos en fotografías de simulación. La definición de las clases proviene del conjunto anotado. Una buena detección por sí sola no demuestra autenticidad, exactitud clínica ni aptitud para dispensar.

En VS Code, abre la carpeta raíz del repositorio, selecciona un kernel Python del entorno de visión e instala los paquetes con `python -m pip install -r qa/vision/requirements.txt`. El notebook entrena desde cero para comparar arquitecturas con los mismos datos y parámetros.


## Preparar una versión en Roboflow

1. Usa un proyecto de detección de objetos con imágenes sintéticas o material autorizado. Marca cajas y clases de forma consistente.
2. Separa las imágenes en entrenamiento, validación y prueba. Mantén todas las fotos de un mismo objeto físico o plantilla en un solo grupo para evitar fuga de información.
3. Genera una versión en Roboflow. Aplica los aumentos solo al entrenamiento, por ejemplo variaciones de iluminación o rotaciones leves cuando representen condiciones del laboratorio.
4. Descarga el ZIP en formato YOLO de detección con `data.yaml`, `train`, `valid` y `test`. Descomprímelo en `.test-artifacts/vision-farmatef/roboflow` e indica la versión exportada en la celda siguiente.

[Versiones del dataset en Roboflow](https://docs.roboflow.com/datasets/versions/dataset-versions) · [Entrenamiento en Ultralytics](https://docs.ultralytics.com/modes/train)


In [ ]:
import csv
from datetime import datetime, timezone
from hashlib import sha256
import json
from pathlib import Path
import platform

import yaml

REPO = next(
    (ruta for ruta in (Path.cwd(), *Path.cwd().parents) if (ruta / "manage.py").is_file()),
    None,
)
if REPO is None:
    raise RuntimeError("Ejecuta el notebook desde la carpeta del repositorio en VS Code.")

ARTEFACTOS = REPO / ".test-artifacts" / "vision-farmatef"
DATASET = ARTEFACTOS / "roboflow"
VERSION_ROBOFLOW = ""  # Ej. "envases-simulados v3"
MODELOS = ("yolov8n.yaml", "yolo11n.yaml")
SEMILLA = 42
EPOCAS = 30
TAMANO = 640
LOTE = 8
DISPOSITIVO = "cpu"  # Usa 0 si el entorno tiene una GPU CUDA disponible.


In [ ]:
ruta_yaml = DATASET / "data.yaml"
if not ruta_yaml.is_file():
    raise FileNotFoundError(f"Descomprime el dataset YOLO donde quede disponible {ruta_yaml}")

fuente = yaml.safe_load(ruta_yaml.read_text(encoding="utf-8"))
if not VERSION_ROBOFLOW.strip():
    raise ValueError("Indica el identificador de la versión de Roboflow que exportaste.")
if not fuente.get("names"):
    raise ValueError("El data.yaml debe declarar las clases en names.")

dataset = DATASET.resolve()
extensiones = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
datos = {"path": str(dataset), "names": fuente["names"]}
conteos = {}
huellas = {}

for split in ("train", "val", "test"):
    indicado = fuente.get(split)
    if not isinstance(indicado, str) or not indicado:
        raise ValueError(f"Falta la carpeta de imágenes del split {split} en data.yaml.")

    carpeta = (dataset / indicado.removeprefix("../")).resolve()
    if not carpeta.is_relative_to(dataset) or not carpeta.is_dir():
        raise ValueError(f"Verifica la ruta de {split}: {carpeta}")

    imagenes = sorted(
        archivo for archivo in carpeta.rglob("*") if archivo.suffix.lower() in extensiones and archivo.is_file()
    )
    if not imagenes:
        raise ValueError(f"El split {split} está vacío.")
    for imagen in imagenes:
        huella = sha256(imagen.read_bytes()).hexdigest()
        otro_split = huellas.setdefault(huella, split)
        if otro_split != split:
            raise ValueError(f"Imagen idéntica en {otro_split} y {split}: {imagen.name}")
    datos[split] = str(carpeta)
    conteos[split] = len(imagenes)

fecha = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S-%fZ")
salida = ARTEFACTOS / "resultados" / fecha
salida.mkdir(parents=True)
datos_absolutos = salida / "data.yaml"
datos_absolutos.write_text(yaml.safe_dump(datos, allow_unicode=True), encoding="utf-8")
conteos


## Entrenar y evaluar

Ambos modelos aprenden del mismo entrenamiento y usan la misma validación para seleccionar sus mejores pesos. Las métricas de la comparación usan exclusivamente el conjunto de prueba. Mantén estos datos aparte si ajustas hiperparámetros después de ver los resultados.


In [ ]:
import torch
import ultralytics
from ultralytics import YOLO

parametros = {
    "epochs": EPOCAS, "imgsz": TAMANO, "batch": LOTE, "seed": SEMILLA,
    "device": DISPOSITIVO, "workers": 0, "deterministic": True,
}
metadata = {
    "version_roboflow": VERSION_ROBOFLOW, "imagenes": conteos, "modelos": MODELOS,
    "parametros": parametros, "ultralytics": ultralytics.__version__, "torch": torch.__version__,
    "python": platform.python_version(), "plataforma": platform.platform(),
}
(salida / "ejecucion.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

resultados = []
for modelo in MODELOS:
    nombre = Path(modelo).stem
    red = YOLO(modelo)
    red.train(
        data=str(datos_absolutos), project=str(salida), name=f"{nombre}-train", pretrained=False,
        optimizer="AdamW", patience=EPOCAS, mosaic=0, fliplr=0, translate=0, scale=0,
        hsv_h=0, hsv_s=0, hsv_v=0, **parametros,
    )
    pesos = Path(red.trainer.best)
    if not pesos.is_file():
        raise FileNotFoundError(f"El entrenamiento de {nombre} no dejó pesos de validación.")
    prueba = YOLO(str(pesos)).val(
        data=str(datos_absolutos), split="test", imgsz=TAMANO, batch=LOTE,
        device=DISPOSITIVO, workers=0, plots=True, project=str(salida), name=f"{nombre}-test",
    )
    resultados.append({
        "modelo": nombre,
        "precision": float(prueba.box.mp),
        "recall": float(prueba.box.mr),
        "map50": float(prueba.box.map50),
        "map50_95": float(prueba.box.map),
        "inferencia_ms_por_imagen": float(prueba.speed["inference"]),
    })

with (salida / "comparacion.csv").open("w", encoding="utf-8", newline="") as archivo:
    tabla = csv.DictWriter(archivo, fieldnames=list(resultados[0]))
    tabla.writeheader()
    tabla.writerows(resultados)

resultados


La carpeta de cada ejecución conserva métricas, pesos, gráficas y registro de configuración. Compara mAP50–95, precisión, recall y latencia por imagen junto con los errores por clase. Una diferencia pequeña en un conjunto reducido necesita más ejemplos independientes para interpretarse.
